# DefectLens AI — Model Training & Evaluation

This notebook walks through:
1. Loading configuration
2. Setting up the training pipeline
3. Training DefectClassifier and PatchCore anomaly detector
4. Evaluating on the test split
5. Visualizing results: confusion matrix, ROC curve, anomaly heatmaps

**Run after completing EDA in `01_dataset_eda.ipynb`.**

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path('..') / 'src'))

import torch
import numpy as np
import matplotlib.pyplot as plt
import yaml

from defectlens.data.dataset import MVTecDataset
from defectlens.data.transforms import get_train_transforms, get_val_transforms
from defectlens.models.classifier import DefectClassifier
from defectlens.models.anomaly import PatchCoreAnomalyDetector
from defectlens.training.trainer import DefectClassifierTrainer
from defectlens.evaluation.metrics import compute_classification_metrics
from defectlens.inference.predictor import DefectPredictor

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
DATA_ROOT = Path('../data/mvtec')
CATEGORY = 'bottle'   # Change to your target category

print(f'Device: {device}')
print(f'Category: {CATEGORY}')
print(f'Dataset root: {DATA_ROOT.resolve()}')

In [ ]:
# --- Configuration ---
with open('../configs/train_classifier.yaml', 'r') as f:
    cfg = yaml.safe_load(f)

IMAGE_SIZE = cfg['data']['image_size']
BATCH_SIZE = cfg['data']['batch_size']
EPOCHS     = cfg['training']['epochs']
LR         = cfg['training']['lr']

print(f'Image size: {IMAGE_SIZE}')
print(f'Batch size: {BATCH_SIZE}')
print(f'Epochs: {EPOCHS}')
print(f'LR: {LR}')

In [ ]:
# --- Data Loaders ---
if DATA_ROOT.exists():
    train_tfm = get_train_transforms(IMAGE_SIZE)
    val_tfm   = get_val_transforms(IMAGE_SIZE)

    train_ds = MVTecDataset(root=str(DATA_ROOT), category=CATEGORY, split='train', transform=train_tfm)
    test_ds  = MVTecDataset(root=str(DATA_ROOT), category=CATEGORY, split='test',  transform=val_tfm)

    train_loader = torch.utils.data.DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
    test_loader  = torch.utils.data.DataLoader(test_ds,  batch_size=BATCH_SIZE, shuffle=False)

    print(f'Train samples: {len(train_ds)}')
    print(f'Test  samples: {len(test_ds)}')
else:
    print('[WARN] Dataset not found. Cannot run training notebook. Download MVTec AD first.')
    train_loader = None
    test_loader  = None

In [ ]:
# --- Train Classifier ---
if train_loader:
    model = DefectClassifier(mode='finetune', pretrained=True, num_classes=2).to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=LR)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, patience=3)
    
    trainer = DefectClassifierTrainer(
        model=model,
        train_loader=train_loader,
        val_loader=test_loader,
        optimizer=optimizer,
        device=device,
        output_dir=f'../outputs/{CATEGORY}',
    )
    history = trainer.fit(num_epochs=EPOCHS)

    # Plot training curves
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    axes[0].plot(history['train_loss'], label='Train Loss')
    axes[0].plot(history['val_loss'],   label='Val Loss')
    axes[0].set_title('Loss Curves'); axes[0].legend(); axes[0].grid(True)
    axes[1].plot(history['val_acc'], label='Val Accuracy', color='green')
    axes[1].set_title('Validation Accuracy'); axes[1].legend(); axes[1].grid(True)
    plt.tight_layout()
    plt.savefig(f'../outputs/{CATEGORY}/training_curves.png', dpi=120)
    plt.show()

In [ ]:
# --- Evaluate on Test Split ---
if test_loader and Path(f'../outputs/{CATEGORY}/best_model.pth').exists():
    predictor = DefectPredictor(
        checkpoint_path=f'../outputs/{CATEGORY}/best_model.pth',
        device=device,
    )

    all_labels, all_preds, all_probs = [], [], []
    model.eval()
    with torch.no_grad():
        for batch in test_loader:
            imgs = batch['image'].to(device)
            lbls = batch['label']
            out  = model(imgs)
            probs = out['probs'].cpu().numpy()
            preds = probs.argmax(axis=1)
            all_labels.extend(lbls.numpy().tolist())
            all_preds.extend(preds.tolist())
            all_probs.extend(probs[:, 1].tolist())

    metrics = compute_classification_metrics(all_labels, all_preds, all_probs)
    print(f'Accuracy:  {metrics["accuracy"]:.4f}')
    print(f'Precision: {metrics["precision"]:.4f}')
    print(f'Recall:    {metrics["recall"]:.4f}')
    print(f'F1:        {metrics["f1"]:.4f}')
    print(f'AUROC:     {metrics["auroc"]:.4f}')